# 03 · Binomial Tree Pricing (CRR)
Cox-Ross-Rubinstein binomial tree — European and American options.
Compares CRR against BS and market mid, and quantifies the early exercise premium.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from utils import bs_price, binomial_price, early_exercise_premium, safe_write_html

pio.renderers.default = "notebook"
pd.set_option("display.float_format", "{:.6f}".format)
print("Imports OK.")

## Load data

In [ ]:
opts = pd.read_parquet("data/opts_with_bs.parquet")

expiries    = sorted(opts["expiry"].unique())
demo_expiry = expiries[0]
demo_df     = opts[opts["expiry"] == demo_expiry].copy().reset_index(drop=True)
print(f"Demo expiry: {demo_expiry.date()}  |  rows: {len(demo_df)}")

## Sanity check: CRR European vs BS on one option

With enough steps CRR converges to BS for European options.

In [ ]:
valid = demo_df.dropna(subset=["iv_calc"])
if valid.empty:
    raise RuntimeError("No valid IVs found. Re-run notebook 02.")
sample   = valid.iloc[min(10, len(valid) - 1)]

S, K, T, r = sample["spot"], sample["strike"], sample["T"], sample["r"]
sigma      = sample["iv_calc"]
opt_type   = sample["type"]

bs_val      = bs_price(S, K, T, r, sigma, option_type=opt_type)
bin_eur_200 = binomial_price(S, K, T, r, sigma, steps=200,
                              option_type=opt_type, option_style="european")

print(f"Black-Scholes (European)  : {bs_val:.4f}")
print(f"Binomial 200-step (EUR)   : {bin_eur_200:.4f}")
print(f"Difference                : {abs(bs_val - bin_eur_200):.6f}")

## Apply CRR European to full demo dataset

In [ ]:
def _binomial_eur(row):
    sigma = row["iv_calc"]
    if pd.isna(sigma):
        return np.nan
    return binomial_price(row["spot"], row["strike"], row["T"], row["r"],
                          sigma, steps=100, option_type=row["type"],
                          option_style="european")

demo_df["binomial_eur"] = demo_df.apply(_binomial_eur, axis=1)
print("European binomial computed. NaN count:", demo_df["binomial_eur"].isna().sum())

## European model comparison (BS vs CRR vs Market)

In [ ]:
demo_df["bs_error"]  = demo_df["bs_price"]    - demo_df["mid"]
demo_df["bin_error"] = demo_df["binomial_eur"] - demo_df["mid"]

mae = lambda x: float(np.nanmean(np.abs(x)))
print(f"MAE  BS        : {mae(demo_df['bs_error']):.6f}")
print(f"MAE  CRR (EUR) : {mae(demo_df['bin_error']):.6f}")

display(demo_df[["strike", "mid", "bs_price", "binomial_eur",
                 "bs_error", "bin_error"]].dropna().head(10))

## Error visualisation — CRR European

In [ ]:
plot_df = demo_df.dropna(subset=["bin_error"])

fig = px.scatter(plot_df, x="strike", y="bin_error", color="mid", size="mid",
                 title=f"CRR European — Pricing Error vs Market  ({demo_expiry.date()})",
                 labels={"bin_error": "Price Error (model - mid)", "strike": "Strike"})
fig.add_hline(y=0, line_dash="dash", line_color="grey", opacity=0.6)
fig.show()
safe_write_html(fig, "binomial_eur_error_plot.html")

## American Option Pricing

The only change from European backward induction is one extra line:
at each node we take `max(continuation_value, exercise_value)`.
This represents the holder choosing to exercise immediately versus
holding the option to the next period.

The **early exercise premium** = American price - European price.
This is always >= 0 by no-arbitrage.

Key theoretical results:
- **Calls (no dividends)**: premium ~= 0. Exercising a call early
  sacrifices time value and the interest benefit of deferring the
  strike payment. It is never optimal on a non-dividend paying asset.
- **Puts**: premium > 0, especially for deep ITM puts close to expiry.
  The holder gives up time value but receives K immediately to invest
  at r. When interest on K exceeds remaining time value, early exercise
  is optimal.

In [ ]:
# American pricing on the same sample contract
am_val, eur_val, premium = early_exercise_premium(
    S, K, T, r, sigma, steps=200, option_type=opt_type
)

print(f"Option type          : {opt_type.upper()}")
print(f"American price       : {am_val:.4f}")
print(f"European price       : {eur_val:.4f}")
print(f"Early exercise premium: {premium:.6f}")
print()
if opt_type == "call":
    print("Note: Premium ~= 0 for calls on non-dividend paying assets -- consistent with theory.")
else:
    print("Note: Positive premium indicates some nodes trigger early exercise.")

## Apply American pricing to full dataset

In [ ]:
def _binomial_am(row):
    sigma = row["iv_calc"]
    if pd.isna(sigma):
        return pd.Series({"binomial_am": np.nan, "early_ex_premium": np.nan})
    am, eu, prem = early_exercise_premium(
        row["spot"], row["strike"], row["T"], row["r"],
        sigma, steps=100, option_type=row["type"]
    )
    return pd.Series({"binomial_am": am, "early_ex_premium": prem})

demo_df[["binomial_am", "early_ex_premium"]] = demo_df.apply(_binomial_am, axis=1)
print("American prices computed.")
print(f"Contracts with premium > 0    : {(demo_df['early_ex_premium'] > 0.001).sum()}")
print(f"Contracts with premium > 0.01 : {(demo_df['early_ex_premium'] > 0.01).sum()}")
print(f"Max premium                   : {demo_df['early_ex_premium'].max():.4f}")

## Early exercise premium by strike

In [ ]:
prem_df = demo_df.dropna(subset=["early_ex_premium"]).copy()

fig = px.scatter(prem_df, x="strike", y="early_ex_premium",
                 color="type",
                 color_discrete_map={"call": "#58a6ff", "put": "#f0883e"},
                 size="mid", size_max=14, opacity=0.8,
                 title=f"Early Exercise Premium by Strike  ({demo_expiry.date()})",
                 labels={"early_ex_premium": "American - European ($)",
                         "strike": "Strike", "type": "Type"})
fig.add_hline(y=0, line_dash="dash", line_color="grey", opacity=0.5)
fig.update_layout(template="plotly_dark")
fig.show()
safe_write_html(fig, "early_exercise_premium_plot.html")

## American vs European: side-by-side comparison table

In [ ]:
display_cols = ["strike", "type", "mid", "bs_price",
                "binomial_eur", "binomial_am", "early_ex_premium", "iv_calc"]
table_df = demo_df[display_cols].dropna(subset=["binomial_am"]).copy()
for col in ["mid","bs_price","binomial_eur","binomial_am","early_ex_premium","iv_calc"]:
    table_df[col] = table_df[col].round(4)

display(table_df.rename(columns={
    "strike": "Strike", "type": "Type",
    "mid": "Mid (Mkt)", "bs_price": "BS (EUR)",
    "binomial_eur": "CRR (EUR)", "binomial_am": "CRR (AM)",
    "early_ex_premium": "EE Premium", "iv_calc": "Impl. Vol",
}).head(20))

## Early exercise premium vs moneyness

Theory predicts the premium is largest for deep ITM puts close to expiry.

In [ ]:
put_df = demo_df[(demo_df["type"] == "put") &
                 demo_df["early_ex_premium"].notna()].copy()
put_df["itm_pct"] = (put_df["strike"] - put_df["spot"]) / put_df["spot"] * 100

fig2 = px.scatter(put_df, x="itm_pct", y="early_ex_premium",
                  color="early_ex_premium",
                  color_continuous_scale="Oranges",
                  size="early_ex_premium", size_max=18,
                  title="Put Early Exercise Premium vs Moneyness",
                  labels={"itm_pct": "(K - S) / S  % (positive = ITM put)",
                          "early_ex_premium": "EE Premium ($)"})
fig2.update_layout(template="plotly_dark")
fig2.show()
safe_write_html(fig2, "early_exercise_vs_moneyness.html")

print()
print("Theory check:")
print("  Deep ITM puts (K >> S) should show highest premium.")
print("  Near-ATM and OTM puts should show near-zero premium.")

## Save

In [ ]:
opts = opts.merge(
    demo_df[["contractSymbol", "binomial_eur", "binomial_am",
             "early_ex_premium", "bin_error"]],
    on="contractSymbol", how="left"
)
opts.to_parquet("data/opts_with_binomial.parquet", index=False)
print("Saved -> data/opts_with_binomial.parquet")
print(f"Columns: {list(opts.columns)}")